# The same QAOA optimization on two sampler backends: python and qiskit

`QAOARunnerSampler` in [B02a](B02a_run_qaoa_optimization.ipynb) hid which simulator actually ran the circuits. This notebook takes the
p=1 optimization of [B02a](B02a_run_qaoa_optimization.ipynb), runs it once on each of the two sampler backends `python`
and `qiskit`, and prints one table: backend, setup time, the trials it actually ran and
the time per trial, the best mean energy each reached, the angles it converged to, and whether
those energies agree.

A backend that cannot be initialized or run here is reported in the table with its error
rather than left out.

`python` samples its statevector with a seeded numpy generator; `qiskit`
samples inside `aer` with its own generator, so its energies are not expected to be identical
to python's. "Agree" is therefore defined below, in the cell that sets `energy_tolerance`.

**What to change**: `number_of_qubits` and `seed_cost_hamiltonian` (the instance);
`number_of_function_calls` and `number_of_samples` (the budget every backend gets);
`BACKENDS_TO_COMPARE` (which backends to try).

**Writes**: nothing. This notebook keeps its results in memory, so re-running it leaves no
rows in the result store.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import time
import traceback

import numpy as np
import pandas as pd
from dotenv import load_dotenv

from quapopt.ancillary_functions.presets import generate_random_hamiltonian
from quapopt.circuits import backend_utilities as bck_utils
from quapopt.data_analysis.data_handling import (CoefficientsType,
                                                 CoefficientsDistribution,
                                                 HamiltonianModels,
                                                 LoggingLevel)
from quapopt.optimization.QAOA import QubitMappingType
from quapopt.optimization.QAOA.implementation.QAOARunnerSampler import QAOARunnerSampler
from quapopt.optimization.parameter_setting.variational.QAOAOptimizationRunner import QAOAOptimizationRunner
from quapopt.optimization.parameter_setting.variational.scipy_tools.ScipyOptimizerWrapped import ScipyOptimizerWrapped

#load env variables -- mainly to read IBM account data for qiskit
load_dotenv()

#The qiskit arm below always runs on, and routes against, the local 'aer' simulator. This
#check only reports whether IBM credentials are present.
HAVE_IBM = any(os.getenv(k) for k in ('IBM_TOKEN', 'IBM_CREDENTIALS_PATH', 'IBM_ACCOUNT_NAME'))
if not HAVE_IBM:
    print("IBM credentials absent: using 'aer'")


### The instance

* One instance, shared by every backend, so the only thing that differs between the runs is
  which simulator produced the samples.
* This notebook compares the backends against each other.
* **What to change here**: `number_of_qubits` (small as those are exact statevector simulations) and `seed_cost_hamiltonian`.
* **Result**: `cost_hamiltonian`.


In [ ]:
number_of_qubits = 24
seed_cost_hamiltonian = 1

#The same Sherrington-Kirkpatrick instance B02a optimizes.
cost_hamiltonian = generate_random_hamiltonian(hamiltonian_model=HamiltonianModels.SherringtonKirkpatrick,
                                               number_of_qubits=number_of_qubits,
                                               seed=seed_cost_hamiltonian,
                                               localities=(2,),
                                               coefficients_type=CoefficientsType.CONTINUOUS,
                                               coefficients_distribution=CoefficientsDistribution.Uniform,
                                               coefficients_distribution_properties={'low': -1,
                                                                                     'high': 1,
                                                                                     'step': 1})

print("Class description (cost):", cost_hamiltonian.hamiltonian_class_description)
print("Instance description (cost):", cost_hamiltonian.hamiltonian_instance_description)


### The budget every backend gets, and what "agree" means

* Every backend is given the same optimizer, the same starting point, the same seed, the same
  number of objective-function calls and the same number of shots per call. The comparison is
  only meaningful because all of that is held fixed.
* Since all backends will generally produce different random samples, their energies differ by shot noise even when the statevector is the same. The tolerance below scales that shot noise as ~1/sqrt(s) from the energy's spread over uniformly random bitstrings. That spread is a reference scale, not an upper bound: a QAOA state can spread its energy more widely. It is also possible that the optimizer will not converge to the same point.
* The compared number is the best mean energy over all calls.
* **What to change here**: `number_of_function_calls`, `number_of_samples`, the optimizer
  settings, and `tolerance_in_standard_errors`.
* **Result**: `classical_optimizer` and `energy_tolerance`.


In [ ]:
#Number of objective function calls each backend is given
number_of_function_calls = 200
#Number of measurements per call
#Can be set to np.inf for python backend for exact computation, but not for qiskit.
number_of_samples = 10**4
qaoa_depth = 1
optimizer_seed = 42

classical_optimizer = ScipyOptimizerWrapped(parameters_bounds=[(-np.pi, np.pi),
                                                               (-np.pi / 2, np.pi / 2)],
                                            optimizer_name='COBYQA',
                                            optimizer_kwargs=None,
                                            basinhopping=True,
                                            basinhopping_kwargs={'niter': 3},
                                            starting_point=[0.01] * 2)

#The exact per-shot standard deviation of the energy over uniformly random bitstrings.
per_shot_standard_deviation = float(np.sqrt(sum(coefficient ** 2 for coefficient, _
                                                in cost_hamiltonian.hamiltonian_list_representation)))
standard_error_of_the_mean = per_shot_standard_deviation / np.sqrt(number_of_samples)

tolerance_in_standard_errors = 3
energy_tolerance = tolerance_in_standard_errors * standard_error_of_the_mean

print(f"Per-shot standard deviation of the energy over uniform bitstrings: {per_shot_standard_deviation:.4f}")
print(f"Standard error of a {number_of_samples}-shot mean at that spread: {standard_error_of_the_mean:.4f}")
print(f"two backends agree when their best mean energies differ by less than "
      f"{energy_tolerance:.4f} ({tolerance_in_standard_errors} standard errors)")


### How each backend is set up

* Every backend is reached through the same `QAOARunnerSampler`; only the
  `initialize_backend_*` call differs. The functions below are those calls, one per backend.
* `python` is the library's own statevector simulator -- numpy on the CPU, cupy on the GPU
  when the instance is large enough to be worth it.
* `qiskit` compiles the ansatz and runs it on the local `aer` simulator.
* **What to change here**: `BACKENDS_TO_COMPARE`, to try a subset or a different order.
* **Result**: `BACKEND_INITIALIZERS`, a name-to-callable mapping.


In [ ]:
def initialize_python(sampler):
    sampler.initialize_backend_python(backend='auto')


def initialize_qiskit(sampler):
    qiskit_backend = bck_utils.get_qiskit_backend(backend_name='aer')
    qiskit_pass_manager, _ = bck_utils.get_qiskit_pass_manager(
        qiskit_backend=qiskit_backend,
        qubit_mapping_type=QubitMappingType.sabre,
        pass_manager_kwargs=None)
    sampler.initialize_backend_qiskit(simulation=True,
                                      qiskit_backend=qiskit_backend,
                                      qiskit_pass_manager=qiskit_pass_manager,
                                      noiseless_simulation=True,
                                      qaoa_depth=qaoa_depth,
                                      qubit_indices_physical=None,
                                      classical_indices=None)


BACKEND_INITIALIZERS = {
                        'qiskit': initialize_qiskit,
                        'python': initialize_python,
                        }

BACKENDS_TO_COMPARE = list(BACKEND_INITIALIZERS)
print("backends to try:", BACKENDS_TO_COMPARE)


### Run the same optimization once per backend

* Each iteration builds a fresh sampler, initializes one backend, runs the optimization, and
  records the setup time, the optimization time, the trials that actually ran, the best mean
  energy and the angles the optimizer converged to.
* Setup is timed apart from the optimization, because it is a real cost a reader pays --
  qiskit's transpilation in particular -- but it is paid once, whatever the budget.
* `number_of_function_calls` is a budget, not a promise: a basinhopping optimizer stops on its
  own criteria and may overshoot or undershoot it, so the time per trial is computed from the
  trials the run actually made.
* A backend that cannot be imported, cannot be initialized, or refuses to sample is caught
  here: its row carries the exception type and message, and the loop moves on.
* **Result**: `backend_rows`, one dictionary per backend.

In [ ]:
backend_rows = []

for backend_label in BACKENDS_TO_COMPARE:
    setup_start_time = time.perf_counter()
    try:
        sampler = QAOARunnerSampler(hamiltonian_representations_cost=[cost_hamiltonian],
                                    hamiltonian_representations_phase=None,
                                    store_n_best_results=1,
                                    store_full_information_in_history=False,
                                    numpy_rng_sampling=np.random.default_rng(seed=0),
                                    logging_level=LoggingLevel.NONE)
        BACKEND_INITIALIZERS[backend_label](sampler)
        optimizer = QAOAOptimizationRunner(qaoa_runner=sampler)
        setup_time = time.perf_counter() - setup_start_time

        optimization_start_time = time.perf_counter()
        best_results, optimization_results = optimizer.run_optimization(
            qaoa_depth=qaoa_depth,
            number_of_function_calls=number_of_function_calls,
            number_of_samples=number_of_samples,
            classical_optimizer=classical_optimizer,
            optimizer_seed=optimizer_seed,
            measurement_noise=None,
            verbosity=0,
            show_progress_bar=False)
        optimization_time = time.perf_counter() - optimization_start_time

        #One row per objective-function call the optimizer actually made.
        trials_run = len(optimization_results.trials_dataframe)
        backend_rows.append({'Backend': backend_label,
                             'SetupTime_s': round(setup_time, 1),
                             'OptimizeTime_s': round(optimization_time, 1),
                             'TrialsRun': trials_run,
                             'SecondsPerTrial': round(optimization_time / trials_run, 3),
                             'BestMeanEnergy': float(optimization_results.best_value),
                             'BestBitstringEnergy': float(best_results[0][0]),
                             'BestAngles': np.round(np.array(optimization_results.best_arguments,
                                                             dtype=float), 4),
                             'Status': 'ran'})
        print(f"{backend_label}: setup {setup_time:.1f} s, then {trials_run} trials in "
              f"{optimization_time:.1f} s ({optimization_time / trials_run:.3f} s/trial), "
              f"best mean energy {optimization_results.best_value:.6f}")

    except Exception as error:
        backend_rows.append({'Backend': backend_label,
                             'SetupTime_s': round(time.perf_counter() - setup_start_time, 1),
                             'OptimizeTime_s': np.nan,
                             'TrialsRun': np.nan,
                             'SecondsPerTrial': np.nan,
                             'BestMeanEnergy': np.nan,
                             'BestBitstringEnergy': np.nan,
                             'BestAngles': np.nan,
                             'Status': f"{type(error).__name__}: {error}"})
        print(f"{backend_label}: NOT COMPARED -- {type(error).__name__}: {error}")
        print('   raised at: ' + traceback.format_exc().strip().splitlines()[-2].strip())

### The table

* One row per backend. `Status` is `ran`, or the exception that stopped it.
* `SetupTime_s` is the one-off cost of building the sampler and its backend; `OptimizeTime_s`
  covers the optimization alone, `TrialsRun` is how many objective-function calls the
  optimizer actually made, and `SecondsPerTrial` divides one by the other. The per-trial time
  is the number to compare, because the backends do not all stop at the same trial.
* `BestAngles` is the point each optimizer converged to. Backends that disagree on the energy
  but converged to different angles disagree about the optimum, not about the simulation.
* `AgreesWithFirst` compares each backend's best mean energy with the first one that ran,
  against the `energy_tolerance` defined above. A backend that did not run carries no verdict.
* **Result**: `comparison_table`, printed.

In [ ]:
comparison_table = pd.DataFrame(backend_rows)

ran = comparison_table[comparison_table['Status'] == 'ran']
if len(ran) == 0:
    raise RuntimeError("no sampler backend ran; nothing to compare")

reference_backend = ran.iloc[0]['Backend']
reference_energy = ran.iloc[0]['BestMeanEnergy']


def agreement_verdict(row):
    if row['Status'] != 'ran':
        return 'not compared'
    difference = abs(row['BestMeanEnergy'] - reference_energy)
    return f"yes ({difference:.4f})" if difference < energy_tolerance else f"NO ({difference:.4f})"


comparison_table['EnergyDiffFrom_' + reference_backend] = comparison_table.apply(
    lambda row: np.nan if row['Status'] != 'ran' else row['BestMeanEnergy'] - reference_energy,
    axis=1)
comparison_table['AgreesWithFirst'] = comparison_table.apply(agreement_verdict, axis=1)

#Printed in two halves only so that the lines stay readable.
timing_columns = ['Backend', 'SetupTime_s', 'OptimizeTime_s', 'TrialsRun', 'SecondsPerTrial',
                  'Status']
energy_columns = ['Backend', 'BestMeanEnergy', 'BestBitstringEnergy', 'BestAngles',
                  'EnergyDiffFrom_' + reference_backend, 'AgreesWithFirst']

print(f"reference backend: {reference_backend} (best mean energy {reference_energy:.6f})")
print(f"tolerance: {energy_tolerance:.4f} "
      f"({tolerance_in_standard_errors} standard errors of a {number_of_samples}-shot mean)")
print(f"{len(ran)} of {len(comparison_table)} backends ran, "
      f"each given a budget of {number_of_function_calls} function calls.")
print()
print(comparison_table[timing_columns].to_string(index=False))
print()
print(comparison_table[energy_columns].to_string(index=False))
comparison_table